# Freeway DQN, Colab training runner

This notebook clones the project repo and runs training, evaluation, the policy comparison, and gameplay recording on a GPU runtime. All code lives in the repo, not in this notebook. This notebook only calls it.

Before running: Runtime menu, Change runtime type, select GPU.

## 1. Setup: clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/kelvintawe12/Freeway-DQN_formative3.git"

# Derive the checkout directory from the repo name so this can't drift
# from REPO_URL again: the repo clones into "Freeway-DQN_formative3".
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# Idempotent: always start from /content and only clone if the repo is not
# already there. Re-running this cell (or running it after a previous run
# left you cd'd inside the repo) will NOT create a nested
# Freeway-DQN_formative3/Freeway-DQN_formative3 checkout. To pull fresh code,
# either delete the folder first or `git -C {REPO_DIR} pull`.
os.chdir("/content")
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL
%cd /content/{REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import os

# Mount Google Drive and send all run outputs there. Every script reads
# FREEWAY_OUTPUT_ROOT via config.py, so models/logs/plots/videos and the
# experiment log are written straight to Drive as training runs. Nothing to
# download afterwards, and a disconnect mid-sweep still leaves every finished
# run's files safe on Drive. The folder syncs to your laptop via the Google
# Drive app or drive.google.com.
#
# To train to the Colab VM instead (not saved on disconnect), skip this cell.
from google.colab import drive
drive.mount("/content/drive")

os.environ["FREEWAY_OUTPUT_ROOT"] = "/content/drive/MyDrive/Freeway-DQN"
os.makedirs(os.environ["FREEWAY_OUTPUT_ROOT"], exist_ok=True)
print("Outputs will be written to:", os.environ["FREEWAY_OUTPUT_ROOT"])

In [ ]:
# Headless rendering support. Needed for anything that opens a display,
# not needed for --record, which captures frames directly.
!apt-get install -y xvfb > /dev/null 2>&1
!pip install -q pyvirtualdisplay

from pyvirtualdisplay import Display
display = Display(visible=0, size=(400, 300))
display.start()

## 2. Sanity check: environment loads

`ale-py` ships the Atari ROMs inside the wheel, so there is no separate ROM-license step. The scripts register the `ALE/*` envs via `config.py` (which calls `gym.register_envs(ale_py)` on import); this standalone cell registers them itself so it works before any script runs.

In [ ]:
import ale_py
import gymnasium as gym

# ale-py bundles the ROMs but does not auto-register with Gymnasium; do it
# explicitly here. The scripts get this for free by importing config.py.
gym.register_envs(ale_py)

env = gym.make("ALE/Freeway-v5")
obs, info = env.reset()
print("ale-py:", ale_py.__version__)
print("Observation shape:", obs.shape)
print("Action space:", env.action_space)
env.close()

## 3. GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4. Smoke test: validate the full pipeline before committing GPU time

This runs a tiny training job (2000 timesteps, low learning-starts so a
handful of real gradient steps happen, not just setup) followed by a
short evaluation and a one-episode recorded clip. The goal is not a
trained agent, two thousand steps will not learn anything meaningful.
The goal is confirming that training, saving, loading, evaluating, and
video recording all execute without error on this specific Colab
runtime, before spending real time on the ten-experiment sweep below.

If this cell fails, fix it here first. The most likely cause is a version
mismatch between the pinned packages in `requirements.txt` and Colab's
preinstalled torch/CUDA build. ALE ROM registration is handled once in
`config.py` (`gym.register_envs(ale_py)`), so every script picks it up on
import — you should not need to patch it in per-file.

In [ ]:
!python train.py \
  --run-id smoke_test \
  --member dev \
  --total-timesteps 2000 \
  --buffer-size 2000 \
  --learning-starts 200 \
  --eval-freq 1000 \
  --n-eval-episodes 2 \
  --notes "smoke test, not a real experiment"

In [ ]:
# ${FREEWAY_OUTPUT_ROOT:-.} resolves to the Drive folder if you mounted it,
# else the repo. The model lives wherever train.py wrote it.
!python evaluate.py --model "${FREEWAY_OUTPUT_ROOT:-.}/models/smoke_test.zip" --episodes 2

In [ ]:
!python play.py --model "${FREEWAY_OUTPUT_ROOT:-.}/models/smoke_test.zip" --record --episodes 1 --video-name smoke_test_clip

If all three cells above completed without a traceback and printed a
reward number, the pipeline is confirmed working end to end. Proceed to
the baseline run and the full sweep below. If something failed, resolve
it now, the sweep uses the exact same code path with a longer budget.

## 5. Baseline training run

This is the shared reference point every member's sweep is measured against.

In [ ]:
!python train.py --run-id baseline --member team --promote \
  --notes "shared reference point for all three sweeps"

## 6. Hyperparameter sweep, 30 experiments across 3 members

Each member's ten runs vary one hyperparameter family from the baseline above, so the effect of each change is readable on its own rather than tangled up with unrelated changes. Every run is a named preset defined in `config.py` (`PRESETS`), so each cell below is one command per experiment. Full reasoning and predicted behavior for every run lives in `EXPERIMENTS.md`.

**Budget:** the sweep runs at `--total-timesteps 150000`, not the 500k default — see the "Compute budget" section of `EXPERIMENTS.md` for why. The full 500k is reserved for the baseline above and the final combined run. Relative hyperparameter ranking is visible well before 500k, and 30 runs at 500k does not fit in a Colab session. Split the three member cells across members/sessions; the shared `experiment_log.csv` merges them automatically.

`member1`, `member2`, `member3` are placeholders. Update the `member` field in `config.py`'s `PRESETS` dictionary once the group assigns real names, then these cells need no other changes.

### Member 1: learning rate

Question this sweep answers: how sensitive is training to the step size of each gradient update, and where does it break down at the extremes?

In [ ]:
M1_PRESETS = [
    "m1_lr_01_tiny", "m1_lr_02_verylow", "m1_lr_03_low", "m1_lr_04_baseline",
    "m1_lr_05_modhigh", "m1_lr_06_high", "m1_lr_07_veryhigh", "m1_lr_08_extreme",
    "m1_lr_09_extreme2", "m1_lr_10_lowfixed",
]

SWEEP_TIMESTEPS = 150000  # reduced budget for the sweep; see EXPERIMENTS.md

for preset in M1_PRESETS:
    !python train.py --preset {preset} --total-timesteps {SWEEP_TIMESTEPS} --notes "TODO: fill in after run"

### Member 2: gamma and batch size

Question this sweep answers: how does the discount factor change the agent's willingness to wait for a safe gap versus rushing, and how does batch size trade off gradient noise against compute cost per update?

In [ ]:
M2_PRESETS = [
    "m2_gamma_01_short", "m2_gamma_02_shortmed", "m2_gamma_03_baseline",
    "m2_gamma_04_long", "m2_gamma_05_verylong",
    "m2_batch_01_small", "m2_batch_02_baseline", "m2_batch_03_mod",
    "m2_batch_04_large", "m2_batch_05_verylarge",
]

SWEEP_TIMESTEPS = 150000  # reduced budget for the sweep; see EXPERIMENTS.md

for preset in M2_PRESETS:
    !python train.py --preset {preset} --total-timesteps {SWEEP_TIMESTEPS} --notes "TODO: fill in after run"

### Member 3: exploration schedule

Question this sweep answers: what is the right balance between exploring the environment early and exploiting a learned policy, and what happens at both extremes of that tradeoff?

In [ ]:
M3_PRESETS = [
    "m3_eps_01_fastdecay", "m3_eps_02_baseline", "m3_eps_03_slowdecay",
    "m3_eps_04_veryslow", "m3_eps_05_highfloor", "m3_eps_06_lowfloor",
    "m3_eps_07_zerofloor", "m3_eps_08_lowstart", "m3_eps_09_alwaysexplore",
    "m3_eps_10_aggressive",
]

SWEEP_TIMESTEPS = 150000  # reduced budget for the sweep; see EXPERIMENTS.md

for preset in M3_PRESETS:
    !python train.py --preset {preset} --total-timesteps {SWEEP_TIMESTEPS} --notes "TODO: fill in after run"

After all thirty runs finish, open `EXPERIMENTS.md` and fill in the Mean Reward and Actual Observed Behavior columns from the log printed below, then write the final combined run using each member's best value (see the Final Combined Run section of `EXPERIMENTS.md`).

## 7. Inspect the experiment log so far

In [ ]:
import os
import pandas as pd

# Read the log from wherever runs wrote it (Drive if mounted, else the repo).
csv_path = os.path.join(os.environ.get("FREEWAY_OUTPUT_ROOT", "."),
                        "experiments", "experiment_log.csv")
log = pd.read_csv(csv_path)
log.sort_values("mean_reward", ascending=False)

## 8. Promote the best run

Once you know which run_id performed best, save its model as `models/dqn_model.zip`, which is what `play.py` and `evaluate.py` load by default.

In [ ]:
import os
import shutil

BEST_RUN_ID = "m1_lr_06_high"  # replace with whichever run_id actually won

# Promote within whichever models dir the runs wrote to (Drive or repo), so
# the source checkpoint actually exists and play.py/evaluate.py find the
# result at the same OUTPUT_ROOT.
models_dir = os.path.join(os.environ.get("FREEWAY_OUTPUT_ROOT", "."), "models")
shutil.copy(os.path.join(models_dir, f"{BEST_RUN_ID}.zip"),
            os.path.join(models_dir, "dqn_model.zip"))
print(f"Promoted {BEST_RUN_ID} to {models_dir}/dqn_model.zip")

## 9. Evaluate the promoted model

In [ ]:
!python evaluate.py --model "${FREEWAY_OUTPUT_ROOT:-.}/models/dqn_model.zip" --episodes 20

## 10. CnnPolicy vs MlpPolicy comparison

In [ ]:
!python compare_policies.py --timesteps 200000

In [ ]:
import os
from IPython.display import Image

Image(os.path.join(os.environ.get("FREEWAY_OUTPUT_ROOT", "."),
                    "plots", "mlp_vs_cnn.png"))

## 11. Record gameplay: before and after training

Recording uses `render_mode="rgb_array"` internally, so it works headlessly and does not depend on the virtual display set up above.

In [ ]:
# Before training: an untrained model, for contrast. Save an initial
# checkpoint with one timestep first (writes to OUTPUT_ROOT/models).
!python train.py --run-id untrained --total-timesteps 1 --notes "untrained baseline for before/after video"
!python play.py --model "${FREEWAY_OUTPUT_ROOT:-.}/models/untrained.zip" --record --episodes 2 --video-name freeway_before_training

## 12. Results are already on Drive

If you ran the Drive-mount cell at the top, there is nothing to package or download: every model, log, plot, video, and the experiment log were written to `/content/drive/MyDrive/Freeway-DQN/` as training ran, and have already synced to your laptop via the Google Drive app or drive.google.com.

The cell below is a **fallback** for sessions where you skipped the Drive mount (training to the Colab VM instead). It zips the VM-local outputs so you can download them before the session ends. It splits the download into a small metrics zip (grab this always, extract the whole thing) and a large models zip (grab only the checkpoints you need), so you can't pull the models down and forget the logs.

In [ ]:
# FALLBACK ONLY — skip if you used the Drive mount (results are already on Drive).
# Zips VM-local outputs from FREEWAY_OUTPUT_ROOT (defaults to the repo).
import os
root = os.environ.get("FREEWAY_OUTPUT_ROOT", ".")

# Light: the metrics you always want. Few MB. Download and extract the WHOLE thing.
!cd "$root" && zip -r /content/results_light.zip logs plots experiments/experiment_log.csv

# Heavy: the 27 MB-each checkpoints. Download only the runs you actually demo.
!cd "$root" && zip -r /content/models.zip models videos

from google.colab import files
files.download("/content/results_light.zip")
# files.download("/content/models.zip")   # uncomment if you need the models locally

In [ ]:
!zip -r results.zip models logs plots videos experiments/experiment_log.csv

from google.colab import files
files.download("results.zip")